In [2]:
"""
datageneratorrq1costcomfort.py

Purpose-built generator for the updated RQ1 cost-comfort scatter figure.
Mirrors the exact RQ1 setup in datagenerator.py (populations, N=500, 3 variants,
10 seeds, 30 days) but captures TWO normalized cost values per group per run:

  cost_norm_mean    -> mean of individual_cost_normalized across ALL 30 days
                       (same as cost_norm_mean in rq1_run_summaries.csv)
  cost_norm_day29   -> individual_cost_normalized on day 29 ONLY
                       (new - shows where agents ended up, not their average)

Output:
  results/rq1costcomfortfigdata.csv
  -> one row per (pop_label, network_code, seed, dominant_group)
  -> 4 populations x 3 variants x 10 seeds x 3 groups = 360 rows
"""

import sys
import numpy as np
import pandas as pd
from pathlib import Path

#point to the project root so the simulation files can be imported
project_root = Path("..")
sys.path.insert(0, str(project_root))

#import run_model and the default epsilons from agent.py
from run_model import run_model
from agent import default_epsilon_habit, default_epsilon_price, default_epsilon_social

#output directory - same as datagenerator.py
results_dir = Path("results")
results_dir.mkdir(parents=True, exist_ok=True)

print("output directory ready")

#RQ1 population compositions - identical to datagenerator.py
#each entry is [habit_pct, price_pct, social_pct] and sums to 100
populations_rq1 = {
    "Habitual":    [90, 5, 5],
    "Progressive": [60, 30, 10],
    "Tipping":     [70, 5, 25],
    "Balanced":    [50, 25, 25]}

#only N=500 for RQ1 - same as datagenerator.py
net_n = 500

#three network variants and ten seeds - same as datagenerator.py
variants = ["a", "b", "c"]
seeds = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

#total simulated days - same as datagenerator.py
days = 30

#total number of runs for progress display
#  -> 4 populations x 3 variants x 10 seeds = 120 runs
#  -> each run produces 3 rows (one per behavioral group)
total_runs = len(populations_rq1) * len(variants) * len(seeds)

print("planned runs:", total_runs)
print("planned rows:", total_runs * 3, "(3 groups per run)")


#running a single RQ1 simulation - identical signature to datagenerator.py run_one()
def run_one(agents_pct, network_code, seed):
    #always use the real epsilons - no baseline needed for RQ1
    df_agents, df_daily, load_profiles, df_pricing = run_model(
        agents_pct = agents_pct,
        network_code = network_code,
        days = days,
        graphs = None,
        median_plot = False,
        random_state = seed,
        epsilon_habit = default_epsilon_habit,
        epsilon_price = default_epsilon_price,
        epsilon_social = default_epsilon_social)

    return df_agents


#computing per-group summary rows for one simulation run
#produces three rows (one per behavioral group) per run
#CHANGED: added cost_norm_day29 column - normalized cost on day 29 only per group
def compute_rq1cc_rows(pop_label, network_code, seed, df_agents):
    #list to collect output rows
    rows = []

    df_active = df_agents.copy()

    #build a per-agent summary with both all-days mean and day-29 cost
    per_agent_list = []
    grouped = df_active.groupby("agent_id")
    for agent_id, agent_data in grouped:
        #dominant_group is the same on every row for one agent, first is safe
        group_name = agent_data["dominant_group"].iloc[0]

        #all-days mean of normalized cost (same computation as datagenerator.py)
        cost_mean_all = agent_data["individual_cost_normalized"].mean()

        #day-29 normalized cost - last day only
        agent_last = agent_data[agent_data["day"] == days - 1]
        if len(agent_last) > 0:
            cost_day29 = float(agent_last["individual_cost_normalized"].iloc[0])
        else:
            #safety fallback - should not happen in a complete run
            cost_day29 = float("nan")

        #day-29 adjustment - individual_adjustment is NaN on all other days
        if len(agent_last) > 0:
            adj = float(agent_last["individual_adjustment"].iloc[0])
        else:
            adj = float("nan")

        agent_row = {
            "agent_id": agent_id,
            "dominant_group": group_name,
            "cost_norm_alldays": cost_mean_all,
            "cost_norm_day29":   cost_day29,
            "individual_adjustment": adj}
        per_agent_list.append(agent_row)

    per_agent = pd.DataFrame(per_agent_list)

    #produce one summary row per behavioral group
    for group in ["Habit-driven", "Price-responsive", "Social-influenced"]:
        g = per_agent[per_agent["dominant_group"] == group]

        #skip if this group has no agents in this run (unusual but safe)
        if len(g) == 0:
            continue

        row_dict = {
            "pop_label":        pop_label,
            "network_code":     network_code,
            "N":                int(network_code[:-1]),
            "seed":             seed,
            "dominant_group":   group,
            #mean cost across all 30 days (same as cost_norm_mean in rq1_run_summaries.csv)
            "cost_norm_mean":   g["cost_norm_alldays"].mean(),
            #CHANGED: new column - mean of day-29 normalized cost across all agents in this group
            "cost_norm_day29":  g["cost_norm_day29"].mean(),
            #day-29 adjustment hours from initial schedule (same as adjustment_mean in rq1_run_summaries.csv)
            "adjustment_mean":  g["individual_adjustment"].mean()}
        rows.append(row_dict)

    return rows


#output CSV path
output_csv = results_dir / "rq1costcomfortfigdata.csv"

#load any already completed runs so the script can resume after interruption
completed_keys = set()
if output_csv.exists():
    df_existing = pd.read_csv(output_csv)
    for i in range(len(df_existing)):
        key = (df_existing.iloc[i]["pop_label"],
               df_existing.iloc[i]["network_code"],
               int(df_existing.iloc[i]["seed"]))
        completed_keys.add(key)
    print("existing output found with", len(completed_keys), "completed runs - will resume")

#run the grid
done_count = 0

for pop_label, agents_pct in populations_rq1.items():
    for variant in variants:
        network_code = str(net_n) + variant
        for seed in seeds:
            done_count = done_count + 1

            #skip if already done from a previous run
            if (pop_label, network_code, seed) in completed_keys:
                print("[" + str(done_count) + "/" + str(total_runs) + "] "
                      + pop_label + " | " + network_code + " | seed " + str(seed)
                      + " -> already done")
                continue

            print("[" + str(done_count) + "/" + str(total_runs) + "] "
                  + pop_label + " | " + network_code + " | seed " + str(seed))

            #run the simulation and extract the agent-level data
            df_a = run_one(agents_pct, network_code, seed)

            #compute the three group-level summary rows
            rows = compute_rq1cc_rows(pop_label, network_code, seed, df_a)

            #append immediately so progress is preserved on interruption
            rows_df = pd.DataFrame(rows)
            header_needed = not output_csv.exists()
            rows_df.to_csv(output_csv, mode = "a", header = header_needed, index = False)
            completed_keys.add((pop_label, network_code, seed))

print("done ->", output_csv)

#quick verification print
df_out = pd.read_csv(output_csv)
print("total rows written:", len(df_out))
print(df_out.groupby(["pop_label", "dominant_group"]).size().rename("n_runs").to_string())

output directory ready
planned runs: 120
planned rows: 360 (3 groups per run)
[1/120] Habitual | 500a | seed 1
Loading network 500a from networks.json...
->  500 agents loaded.
  Group split: 450 Habit-driven, 25 Price-responsive, 25 Social-influenced  (total: 500)
Initialising agents...

Running simulation: 30 days, 500 agents, seed 1
  epsilon_habit=1.0  epsilon_price=0.25  epsilon_social=0.5

Day 1 / 30 | peak: 478.76 kW | mean: 314.62 kW | PAR: 1.52 | flex: 0.00 | norm_flex: 0.00 | price_mean: 7.73
Day 2 / 30 | peak: 478.09 kW | mean: 315.35 kW | PAR: 1.52 | flex: 943.18 | norm_flex: 1.89 | price_mean: 7.70
Day 3 / 30 | peak: 514.06 kW | mean: 318.32 kW | PAR: 1.61 | flex: 940.99 | norm_flex: 1.88 | price_mean: 7.70
Day 4 / 30 | peak: 514.47 kW | mean: 321.05 kW | PAR: 1.60 | flex: 926.97 | norm_flex: 1.85 | price_mean: 7.72
Day 5 / 30 | peak: 511.93 kW | mean: 317.49 kW | PAR: 1.61 | flex: 918.22 | norm_flex: 1.84 | price_mean: 7.77
Day 6 / 30 | peak: 515.29 kW | mean: 315.95 kW |